### **Student Information**
GitHub ID: wenting2110

Kaggle name: wenting2110

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

## 1. Model Development (10 pts Required)

### 1.1 Preprocessing Steps

The competition data comes in three files, merged on `post_id`: 
* `final_posts.json` (the raw text)
* `data_identification.csv` (train/test split)
* `emotion.csv` (labels for the 3,360 training posts)

The 1,440 test posts have no labels and are what the model predicts.

Before building features, we need to reduce noise and merge surface variants of the same word, so each post is cleaned with a single `clean_text` function that: 
* Lowercases the text, so Happy and happy map to one token
* Removes URLs and @mentions, which carry no emotion signal
* Keeps only letters, digits and apostrophes, dropping other punctuation
* Strips repeated whitespace

### 1.2 Feature Engineering Steps

Each post is represented with two complementary feature sets, and they are concatenated into one vector.

* **TF-IDF (surface/lexical signal)**
  A word n-gram vectorizer (`ngram_range=(1,2)`, `min_df=2`, `sublinear_tf=True`) combined with a character n-gram vectorizer (`analyzer="char_wb"`, `ngram_range=(2,5)`). Word features capture vocabulary, and character features capture sub-word patterns such as prefixes, suffixes and non-standard spelling, which matter on short, noisy posts.

* **MiniLM sentence embeddings (semantic signal)**
  Each post is encoded with the pre-trained `all-MiniLM-L6-v2` sentence-transformer into a normalized dense vector. Unlike TF-IDF, this captures meaning, so two posts expressing the same emotion with different words land close together.

The sparse TF-IDF matrix and the dense MiniLM matrix are joined with `scipy.sparse.hstack`. Giving the classifier both a lexical view and a semantic view was the single largest improvement in the experiments.

### 1.3 Explanation of Your Model

Candidates were compared using 5-fold stratified cross-validation (macro F1) to guide model selection without overfitting to the test set.  
The progression was:
* Word-only TF-IDF with a Linear SVM scored 0.489 (below the 0.5 baseline).
* Adding character n-grams and switching to a balanced Logistic Regression classifier raised the score to about 0.50.
* Concatenating MiniLM embeddings increased the score to 0.5525.

The final model is a Logistic Regression classifier (`class_weight="balanced"`, `C=1`, `max_iter=3000`):

* **Logistic Regression classifier**: Handles the combined high-dimensional sparse-plus-dense feature matrix effectively, outputs class probabilities, and trains quickly on CPU.
 
* **`class_weight="balanced"`**: Addresses class imbalance (e.g., `joy` has 875 training posts while `fear` has 288). Without balancing, rare classes receive less weight and macro F1 drops. Balancing upweights minority classes to improve per-class recall and balance.

* **`C=1`**: `C` controls inverse regularization strength, with `C=1` selected for strong generalization. 

* **`max_iter=3000`**: Ensures the solver converges on the combined feature space.



### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I used Claude as an AI assistant while developing the competition model. The main prompts, the techniques it suggested, and why I decided to adopt each were:

* "幫我在這個情緒資料集上建立一個分類模型,可以先試試看用 TF-IDF + linear SVM。"

  It reproduced the TF-IDF + linear SVM system. I adopted it as a starting point because it was the strongest baseline I already understood, but it only scored 0.489 in cross-validation, which is below the 0.5 baseline. So I had to go further.

* "分數有點太低了,該怎麼調整才好?"

  It suggested adding character n-grams and setting `class_weight="balanced"`. I accepted both. Because the metric is macro F1, which weights every class equally, yet my data is imbalanced (joy has 875 training posts, fear only 288), so without balancing the model would neglect the rare classes and drag the macro score down. Character n-grams made sense because the posts are short and noisy, where sub-word patterns carry signal that whole words miss.


* "分數有提升了,不過我們的成績在 leaderboard 上倒數耶,該怎麼加強?"

  It suggested concatenating pre-trained MiniLM sentence embeddings with the TF-IDF features. I adopted this because TF-IDF only sees exact words, so two posts expressing the same emotion in different words look unrelated to it by adding the meaning-based embeddings. This gave the single largest jump, up to 0.5525, so I kept it.

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
### Add the code related to the preprocessing steps in cells inside this section
import json, os, re, warnings; warnings.filterwarnings("ignore")
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer

DATA = "2026-dm-lab-2-private-competition"

# 1. 載入 + 合併
posts = pd.DataFrame(json.load(open(os.path.join(DATA, "final_posts.json"), encoding="utf-8")))
ident = pd.read_csv(os.path.join(DATA, "data_identification.csv"))
emo   = pd.read_csv(os.path.join(DATA, "emotion.csv"))
df = posts.merge(ident, on="post_id").merge(emo, on="post_id", how="left")
train_df = df[df["set"] == "train"].reset_index(drop=True)
test_df  = df[df["set"] == "test"].reset_index(drop=True)

# 2. 前處理
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r"http\S+|www\.\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    s = re.sub(r"[^a-z0-9'\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()
train_df["clean"] = train_df["text"].apply(clean_text)
test_df["clean"]  = test_df["text"].apply(clean_text)

## 2. Feature Engineering Steps

In [2]:
### Add the code related to the feature engineering steps in cells inside this section
# 3. 特徵 A：字 + 字元 TF-IDF
vec = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
])
Xtr_tfidf = vec.fit_transform(train_df["clean"])
Xte_tfidf = vec.transform(test_df["clean"])

# 3. 特徵 B：MiniLM 句向量
st = SentenceTransformer("all-MiniLM-L6-v2")
Etr = csr_matrix(st.encode(train_df["text"].tolist(), normalize_embeddings=True))
Ete = csr_matrix(st.encode(test_df["text"].tolist(),  normalize_embeddings=True))

# 合併兩種特徵
X_train = hstack([Xtr_tfidf, Etr]).tocsr()
X_test  = hstack([Xte_tfidf, Ete]).tocsr()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 3. Model Implementation Steps

In [3]:
### Add the code related to the model implementation steps in cells inside this section
# 4. 模型
clf = LogisticRegression(max_iter=3000, class_weight="balanced", C=1)
clf.fit(X_train, train_df["label"])

# 5. 預測 + 輸出（表頭 id,label）
test_df["label"] = clf.predict(X_test)
test_df[["post_id", "label"]].rename(columns={"post_id": "id"}).to_csv("submission.csv", index=False)
print("done:", len(test_df), "rows")

done: 1440 rows
